In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*ดู [API reference](https://docs.quantum.ibm.com/api/functions/kipu-optimization)*


> **Note:** * Qiskit Functions เป็นฟีเจอร์ทดลองที่ให้บริการเฉพาะผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น ยังอยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง

## ภาพรวม

ด้วย Iskay Quantum Optimizer ของ Kipu Quantum คุณสามารถแก้ปัญหาการปรับให้เหมาะสมที่ซับซ้อนโดยใช้คอมพิวเตอร์ควอนตัมของ IBM&reg; solver นี้ใช้ประโยชน์จาก [bf-DCQO algorithm](https://doi.org/10.48550/arXiv.2409.04477) ล้ำสมัยของ Kipu ที่ต้องการเพียงฟังก์ชันเป้าหมายเป็น input เพื่อส่งมอบคำตอบของปัญหาโดยอัตโนมัติ รองรับปัญหาการปรับให้เหมาะสมที่มี Qubit ได้สูงสุด 156 ตัว ทำให้ใช้ Qubit ทั้งหมดของอุปกรณ์ควอนตัม IBM ได้ Optimizer ใช้การ mapping แบบ 1-to-1 ระหว่างตัวแปรแบบ classical และ Qubit ซึ่งช่วยให้แก้ปัญหาการปรับให้เหมาะสมที่มีตัวแปรไบนารีได้สูงสุด 156 ตัว

Optimizer รองรับการแก้ปัญหา binary optimization แบบไม่มีข้อจำกัด นอกจากการใช้สูตร QUBO (Quadratic Unconstrained Binary Optimization) ทั่วไปแล้ว ยังรองรับปัญหา optimization ลำดับสูง (HUBO) อีกด้วย solver ใช้ quantum algorithm แบบไม่ใช้ variational โดยทำการคำนวณส่วนใหญ่บนอุปกรณ์ควอนตัม

เนื้อหาต่อไปนี้ให้รายละเอียดเพิ่มเติมเกี่ยวกับ algorithm ที่ใช้ คำแนะนำสั้นๆ ในการใช้ฟังก์ชัน รวมถึงผลการ benchmark บนปัญหาหลายขนาดและความซับซ้อน

## คำอธิบาย

Optimizer เป็นการนำ quantum optimization algorithm ล้ำสมัยมาใช้งานได้จริง โดยแก้ปัญหาการปรับให้เหมาะสมด้วยการรัน quantum circuit ที่บีบอัดสูงบน quantum hardware การบีบอัดนี้ทำได้โดยการนำ counterdiabatic term เข้าสู่ time evolution พื้นฐานของระบบควอนตัม algorithm ทำงานหลายรอบบน hardware เพื่อให้ได้คำตอบสุดท้ายและรวมกับ post-processing ขั้นตอนเหล่านี้ถูกรวมไว้ใน workflow ของ Optimizer อย่างราบรื่นและทำงานโดยอัตโนมัติ

### Quantum Optimizer ทำงานอย่างไร?

ส่วนนี้อธิบายพื้นฐานของ bf-DCQO algorithm ที่นำมาใช้ บทนำของ algorithm นี้ยังสามารถหาได้บน [Qiskit YouTube channel](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s).

algorithm นี้อ้างอิงจาก time evolution ของระบบควอนตัมที่เปลี่ยนแปลงตามเวลา โดยคำตอบของปัญหาถูกเข้ารหัสใน ground state ของระบบควอนตัมเมื่อสิ้นสุด evolution ตาม [adiabatic theorem](https://en.wikipedia.org/wiki/Adiabatic_theorem) evolution นี้ต้องช้าเพื่อให้ระบบอยู่ใน ground state การ digitize evolution นี้เป็นพื้นฐานของ digitized quantum adiabatic computation (DQA) และ QAOA algorithm อย่างไรก็ตาม evolution ช้าที่ต้องการนั้นไม่เหมาะสำหรับขนาดปัญหาที่เพิ่มขึ้น เนื่องจากส่งผลให้ circuit depth เพิ่มขึ้น การใช้ counterdiabatic protocol ช่วยลด excitation ที่ไม่ต้องการในช่วง evolution time สั้นในขณะที่ยังคงอยู่ใน ground state การ digitize evolution time ที่สั้นกว่านี้ส่งผลให้ quantum circuit มี depth สั้นลงและมี entangling gate น้อยลง

Circuit ของ bf-DCQO algorithm โดยทั่วไปใช้ entangling gate น้อยกว่า DQA ถึงสิบเท่า และน้อยกว่า QAOA standard implementation สามถึงสี่เท่า เนื่องจาก Gate มีจำนวนน้อยกว่า ข้อผิดพลาดในการรัน circuit บน hardware จึงน้อยลง ดังนั้น optimizer จึงไม่จำเป็นต้องใช้เทคนิคอย่าง error suppression หรือ error mitigation การนำไปใช้ใน version ในอนาคตสามารถเพิ่มคุณภาพของคำตอบได้อีก

แม้ว่า bf-DCQO algorithm จะใช้การวนซ้ำ แต่มันไม่ใช่ variational หลังจากแต่ละรอบของ algorithm การกระจายของ state จะถูกวัด การกระจายที่ได้จะถูกใช้คำนวณ bias-field bias-field ช่วยให้เริ่มรอบถัดไปจาก energy state ใกล้กับคำตอบที่พบก่อนหน้านี้ ด้วยวิธีนี้ algorithm จะเคลื่อนไปยังคำตอบที่มี energy ต่ำกว่าในแต่ละรอบ โดยทั่วไปประมาณสิบรอบเพียงพอสำหรับการ converge ไปยังคำตอบ ซึ่งต้องการจำนวนรอบต่ำกว่า variational algorithm มากที่อยู่ในระดับประมาณ 100 รอบ

optimizer รวม bf-DCQO algorithm กับ classical post-processing หลังจากวัดการกระจายของ state จะทำการค้นหาเฉพาะที่ (local search) ในระหว่างการค้นหา bits ของคำตอบที่วัดได้จะถูกพลิกแบบสุ่ม หลังจากพลิก energy ของ bitstring ใหม่จะถูกประเมิน หาก energy ต่ำกว่า bitstring จะถูกเก็บไว้เป็นคำตอบใหม่ local search ขยายแบบ linear ตามจำนวน Qubit จึงมีค่าใช้จ่ายในการคำนวณต่ำ เนื่องจาก post-processing แก้ไข local bitflip จึงชดเชยข้อผิดพลาด bit-flip ที่มักเกิดจากความไม่สมบูรณ์ของ hardware และ readout error

### Workflow

แผนผังของ workflow ของ Quantum Optimizer มีดังต่อไปนี้

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Workflow ของ Quantum Optimizer")

การใช้ Quantum Optimizer ช่วยลดการแก้ปัญหาการปรับให้เหมาะสมบน quantum hardware ให้เหลือเพียง
* กำหนดสูตรฟังก์ชันเป้าหมายของปัญหา
* เข้าถึง Optimizer ผ่าน Qiskit Functions
* รัน Optimizer และเก็บผลลัพธ์

## Benchmarks

ผลการ benchmark ด้านล่างแสดงให้เห็นว่า Optimizer จัดการปัญหาที่มี Qubit สูงสุด 156 ตัวได้อย่างมีประสิทธิภาพ และให้ภาพรวมทั่วไปของความแม่นยำและการขยายขนาดของ optimizer ในปัญหาประเภทต่างๆ โปรดทราบว่าผลการทำงานจริงอาจแตกต่างกันขึ้นอยู่กับลักษณะเฉพาะของปัญหา เช่น จำนวนตัวแปร ความหนาแน่นและ locality ของ term ในฟังก์ชันเป้าหมาย และลำดับพหุนาม

ตารางต่อไปนี้รวม approximation ratio (AR) ซึ่งเป็นค่าที่กำหนดดังนี้:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
โดยที่ $C$ คือฟังก์ชันเป้าหมาย $C_{\textrm{min}}$, $C_{\textrm{max}}$ คือค่าต่ำสุดและสูงสุดของมัน และ $C^{*}$ คือ cost ของคำตอบที่ดีที่สุดที่พบ ดังนั้น AR=100\% หมายความว่าได้ ground state ของปัญหามาแล้ว

| ตัวอย่าง | จำนวน Qubit | Approximation Ratio | เวลารวม (s) | การใช้ Runtime (s) | จำนวน shot ทั้งหมด | จำนวนรอบ |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- MaxCut instance ที่มี 28, 30 และ 32 Qubit รันบน ibm_sherbrooke ส่วน instance ที่มี 80, 100 และ 120 รันบน Heron r2 processor
- HUBO instance ก็รันบน Heron r2 processor เช่นกัน

benchmark instance ทั้งหมดสามารถเข้าถึงได้บน GitHub (ดู [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)) ตัวอย่างการรัน instance เหล่านี้สามารถพบได้ใน [ตัวอย่างที่ 3: Benchmark instances](#example-3-benchmark-instances)

## เริ่มต้นใช้งาน

ในเอกสารนี้ เราจะผ่านขั้นตอนการใช้ Iskay Quantum Optimizer ในกระบวนการนี้ เราจะแสดงอย่างรวดเร็วว่าจะโหลดฟังก์ชันจาก catalog ได้อย่างไร และวิธีแปลงปัญหาเป็น input ที่ถูกต้อง พร้อมกับแสดงวิธีทดลองกับ optional parameter ต่างๆ

สำหรับตัวอย่างที่ละเอียดกว่า โปรดดู tutorial [แก้ปัญหา Market Split ด้วย Kipu Quantum's Iskay Quantum Optimizer](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer) ซึ่งเราผ่านกระบวนการทั้งหมดของการใช้ Iskay Solver เพื่อแก้ปัญหา Market Split ซึ่งเป็นความท้าทายการจัดสรรทรัพยากรในโลกจริงที่ต้องแบ่งตลาดออกเป็นภูมิภาคการขายที่สมดุลเพื่อตอบสนองเป้าหมายความต้องการที่แน่นอน

ยืนยันตัวตนโดยใช้ API key ของคุณที่พบใน [IBM Quantum Platform dashboard](http://quantum.cloud.ibm.com/) และเลือก Qiskit Function ดังนี้:

In [ ]:
# ruff: noqa: F821

> **Note:** โค้ดต่อไปนี้สมมติว่าคุณได้บันทึก credentials ไว้แล้ว หากยังไม่ได้ ให้ทำตามคำแนะนำใน [บันทึกบัญชี IBM Cloud ของคุณ](/guides/functions#install-qiskit-functions-catalog-client) เพื่อยืนยันตัวตนด้วย API key

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## ตัวอย่างการตั้งค่าแบบกำหนดเอง
ต่อไปนี้คือวิธีการตั้งค่า Iskay ด้วยการตั้งค่าต่างๆ:

In [ ]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

## ตัวอย่างที่ 1: ฟังก์ชัน cost อย่างง่าย
พิจารณาฟังก์ชัน cost ในรูปแบบ spin:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

โดยที่ $(x_0, ..., x_4) \in {-1, 1}^5$

คำตอบของฟังก์ชัน cost อย่างง่ายนี้คือ
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
ที่ค่าต่ำสุด $C^{*} = -6$

### 1. สร้างฟังก์ชันเป้าหมาย
เริ่มต้นด้วยการสร้าง dictionary ที่มีสัมประสิทธิ์ของฟังก์ชันเป้าหมายดังนี้:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. รัน Optimizer
เราแก้ปัญหาโดยรัน optimizer เนื่องจาก $(x_0, ..., x_4) \in {-1, 1}^5$ เราต้องตั้งค่า `problem_type=spin`

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. ดึงผลลัพธ์
คำตอบของปัญหาการปรับให้เหมาะสมถูกให้มาโดยตรงจาก optimizer

In [ ]:
print(job.result())

ซึ่งจะแสดง dictionary ในรูปแบบ:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

สังเกตว่า dictionary `solution` แสดง result vector $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$

## ตัวอย่างที่ 2: MaxCut

ปัญหา graph หลายอย่างเช่น MaxCut หรือ Maximum independent set เป็นปัญหา NP-hard และเป็นตัวเลือกที่เหมาะสำหรับทดสอบ quantum algorithm และ hardware ตัวอย่างนี้แสดงการแก้ปัญหา MaxCut ของ 3-regular graph ด้วย Quantum Optimizer

ในการรันตัวอย่างนี้คุณต้องติดตั้ง package `networkx` นอกเหนือจาก `qiskit-ibm-catalog` ในการติดตั้ง รันคำสั่งต่อไปนี้:

In [ ]:
# %pip install networkx numpy

### 1. สร้างฟังก์ชันเป้าหมาย
เริ่มต้นด้วยการสร้าง random 3-regular graph สำหรับ graph นี้ เรากำหนดฟังก์ชันเป้าหมายของปัญหา MaxCut

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the max-cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. รัน Optimizer
แก้ปัญหาโดยรัน optimizer

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. ดึงผลลัพธ์
ดึงผลลัพธ์และ map bitstring ของคำตอบกลับไปยัง node ของ graph ดั้งเดิม

In [ ]:
print(job.result())

คำตอบของปัญหา Maxcut อยู่ใน sub-dictionary `solution` ของ result object โดยตรง

In [ ]:
maxcut_solution = job.result()["solution"]

## ตัวอย่างที่ 3: Benchmark instances
benchmark instance สามารถเข้าถึงได้บน GitHub: [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances).

instance สามารถโหลดได้โดยใช้ library `pygithub` ในการติดตั้ง รันคำสั่งต่อไปนี้:

In [ ]:
# %pip install pygithub

path สำหรับ benchmark instance คือ:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

เพื่อทำซ้ำประสิทธิภาพของ benchmark สำหรับ HUBO instance ให้เลือก Backend `ibm_marrakesh` และตั้งค่า `direct_qubit_mapping` เป็น `True` ใน sub-dictionary `options`

ตัวอย่างต่อไปนี้รัน Maxcut instance ที่มี 32 node

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_regular_3_150_nodes_weighted.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "<BACKEND-NAME>",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Use case
use case ทั่วไปสำหรับ Optimization solver คือปัญหา combinatorial optimization คุณสามารถแก้ปัญหาจากหลายอุตสาหกรรมเช่น finance, pharmaceutics หรือ logistics ตัวอย่างบางส่วนได้แก่:
* Portfolio optimization (QUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.22.054037) และ [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Protein folding (HUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Logistics scheduling (QUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Network optimization: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split (QUBO): [tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

หากคุณสนใจแก้ปัญหา use case เฉพาะและพัฒนา mapping ที่เหมาะสม เราสามารถช่วยได้ [ติดต่อเรา](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).

## ขอรับการสนับสนุน
สำหรับการสนับสนุน ติดต่อ support@kipu-quantum.com

## ขั้นตอนถัดไป
- [ขอเข้าถึง Quantum Optimizer โดย Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).
- เยี่ยมชม [API reference](https://docs.quantum.ibm.com/api/functions/kipu-optimization) สำหรับ Qiskit Function นี้
- ลอง tutorial [แก้ปัญหา Market Split ด้วย Kipu Quantum's Iskay Quantum Optimizer](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)
- อ่าน [Romero, S. V., et al. (2025). Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477](https://arxiv.org/abs/2409.04477).
- อ่าน [Cadavid, A. G., et al. (2024). Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898](https://arxiv.org/abs/2405.13898).
- อ่าน [Chandarana, P., et al. (2025). Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663](https://arxiv.org/abs/2505.08663).

## ข้อมูลเพิ่มเติม
Iskay เช่นเดียวกับชื่อบริษัทของเรา Kipu Quantum เป็นคำภาษาเปรู แม้ว่าเราจะเป็น startup จากเยอรมนี แต่คำเหล่านี้มาจากประเทศของ co-founder คนหนึ่งของเรา ซึ่ง Quipu เป็นหนึ่งในเครื่องคำนวณแรกๆ ที่มนุษย์พัฒนาขึ้นเมื่อ 2000 ปีก่อนคริสตกาล